Imports and Setup

In [1]:
import pandas as pd
import sys
import os

sys.path.append("..")
from src.retrieval.hybrid_engine import HybridRetrievalEngine
from src.reranker.reranker import NeuralReranker

print("Loading Hybrid Engine and Cross-Encoder Reranker...")
hybrid_engine = HybridRetrievalEngine(
    data_path="../data/books_cleaned.csv", 
    chroma_dir="../data/chroma_db"
)
reranker = NeuralReranker()
print("Engines loaded successfully.")


/Users/simranchaudhary/Documents/ShelfKnows/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/simranchaudhary/Documents/ShelfKnows/venv/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loading Hybrid Engine and Cross-Encoder Reranker...
Initializing Hybrid Retrieval Engines (Vector + BM25 + Collaborative Filtering)...
Initializing Embedding Model ('all-MiniLM-L6-v2')...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Connecting to local ChromaDB vector store at '../data/chroma_db'...
 Building BM25 Sparse Index from '../data/books_cleaned.csv'...
 BM25 Index successfully built for 50000 books in 0.82 seconds!
 Initializing Collaborative Filtering Engine from '../data/books_cleaned.csv'...
 Collaborative Filtering Model trained across 500 books in 1.72 seconds!
All three engines loaded successfully.
Loading Cross-Encoder model ('cross-encoder/ms-marco-MiniLM-L-6-v2')...
Cross-Encoder loaded successfully.
Engines loaded successfully.


Defining the Intelligent Explanation Generator

In [2]:
def generate_recommendation_reason(query, book):
    """
    Generates a personalized, context-aware explanation for why 
    this specific book was recommended for the user's query.
    """
    title = book.get('title', 'Unknown Title')
    authors = book.get('authors', 'Unknown Author')
    genre = book.get('genres', 'Fiction')
    rating = book.get('average_rating', 4.0)
    
    query_lower = query.lower()
    
    # Identify key thematic overlap between query and book
    theme_highlight = ""
    if any(w in query_lower for w in ["survival", "danger", "game"]):
        theme_highlight = "high-stakes survival elements and tension"
    elif any(w in query_lower for w in ["rebellion", "regime", "oppress", "dystopian"]):
        theme_highlight = "themes of political defiance and resistance against power"
    elif any(w in query_lower for w in ["magic", "wizard", "fantasy", "dragon"]):
        theme_highlight = "rich magical world-building and mystical adventure"
    elif any(w in query_lower for w in ["space", "future", "sci-fi", "robot"]):
        theme_highlight = "futuristic exploration and technological intrigue"
    else:
        theme_highlight = f"compelling {genre.lower()} storytelling and thematic depth"

    explanation = (
        f"Recommended for your interest in '{query}': '{title}' by {authors} delivers {theme_highlight}, "
        f"backed by a strong community rating of {rating} stars."
    )
    return explanation

print("Explanation generator function defined successfully.")


Explanation generator function defined successfully.


Testing End-to-End Recommendations with AI Explanations

In [3]:
test_query = "books about high-stakes survival games and rebellion against oppressive regimes"

print(f"Step 1: Running Stage 1 Hybrid Retrieval for: '{test_query}'")
candidates = hybrid_engine.search(test_query, top_k=10)

print("Step 2: Running Stage 2 Neural Cross-Encoder Reranking...")
top_books = reranker.rerank(test_query, candidates, top_k=3)

print("Step 3: Generating Personalized Explanations...\n")
for rank, book in enumerate(top_books, 1):
    reason = generate_recommendation_reason(test_query, book)
    print(f"Rank #{rank}: {book['title']} by {book['authors']}")
    print(f"Genre: {book['genres']} | Rating: {book['average_rating']} stars")
    print(f"Reason: {reason}\n")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Step 1: Running Stage 1 Hybrid Retrieval for: 'books about high-stakes survival games and rebellion against oppressive regimes'
⚡ Vector Search for 'books about high-stakes survival games and rebellion against oppressive regimes' returned 20 matches in 59.74 ms
⚡ BM25 Search for 'books about high-stakes survival games and rebellion against oppressive regimes' returned 20 matches in 100.80 ms
 Book 'Seeds of Rebellion (Beyonders, #2)' not found in top interaction catalog.
Hybrid Retrieval completed for 'books about high-stakes survival games and rebellion against oppressive regimes' in 161.94 ms
Step 2: Running Stage 2 Neural Cross-Encoder Reranking...
Neural Reranking completed for 10 books in 682.52 ms
Step 3: Generating Personalized Explanations...

Rank #1: Seeds of Rebellion (Beyonders, #2) by Brandon Mull
Genre: General Fiction | Rating: 4.25 stars
Reason: Recommended for your interest in 'books about high-stakes survival games and rebellion against oppressive regimes': 'Seeds of 